# Clair — Quantization & Export to GGUF

This notebook merges the Clair LoRA adapters with the base model and exports to GGUF format for CPU inference with llama.cpp.

## Steps:
1. Merge LoRA adapters with base model
2. Export to multiple GGUF quantization formats
3. Test inference speed
4. Select best model for 7GB RAM constraint

## Target Quantizations:
- Q4_K_M: ~1.8GB (recommended for 7GB RAM)
- Q5_K_M: ~2.1GB (better quality, more RAM)
- Q3_K_M: ~1.5GB (smaller, lower quality)

In [ ]:
import os
import torch
from unsloth import FastLanguageModel
import subprocess
import time

# Configuration - use absolute paths
PROJECT_ROOT = "/mnt/workspace/zim-my"
base_model_path = "/mnt/workspace/models/Qwen/Qwen2.5-3B-Instruct"
lora_path = os.path.join(PROJECT_ROOT, "models", "clair-lora")
output_dir = os.path.join(PROJECT_ROOT, "models", "clair-gguf")
max_seq_length = 2048

os.makedirs(output_dir, exist_ok=True)

print(f"Base model: {base_model_path}")
print(f"LoRA adapters: {lora_path}")
print(f"Output directory: {output_dir}")

# Verify paths exist
if not os.path.exists(base_model_path):
    raise FileNotFoundError(f"Base model not found at {base_model_path}")
if not os.path.exists(lora_path):
    raise FileNotFoundError(f"LoRA adapters not found at {lora_path}")
    
print("✓ All paths verified")

## 1. Load Model with LoRA Adapters

In [ ]:
# Load base model first, then apply LoRA adapters
print("Loading base model...")
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=base_model_path,
    max_seq_length=max_seq_length,
    dtype=None,
    load_in_4bit=True,
)

print("✓ Base model loaded")
print("Applying LoRA adapters...")

# Apply LoRA adapters
from peft import PeftModel
model = PeftModel.from_pretrained(model, lora_path)

print("✓ LoRA adapters applied successfully")

## 2. Merge LoRA Adapters

In [ ]:
# Merge LoRA adapters into base model
merged_model_path = os.path.join(PROJECT_ROOT, "models", "merged", "clair")
os.makedirs(merged_model_path, exist_ok=True)

print("Merging LoRA adapters with base model...")
model.save_pretrained_merged(merged_model_path, tokenizer, save_method="merged_16bit")

print(f"✓ Merged model saved to: {merged_model_path}")

# Verify merge
if not os.path.exists(os.path.join(merged_model_path, "model.safetensors")):
    raise FileNotFoundError("Merge failed - model.safetensors not found")
print("✓ Merge verified")

## 3. Export to GGUF Format

In [ ]:
# Export to multiple quantization formats
quantization_methods = [
    ("q4_k_m", "Q4_K_M"),  # Recommended for 7GB RAM
    ("q5_k_m", "Q5_K_M"),  # Better quality
    ("q3_k_m", "Q3_K_M"),  # Smaller size
]

print("Exporting to GGUF formats...\n")

# Use the already-loaded model with LoRA for GGUF export
# Unsloth will handle the merging internally
for method, name in quantization_methods:
    output_file = f"{output_dir}/clair-{name.lower()}.gguf"
    print(f"Exporting to {name}...")
    
    model.save_pretrained_gguf(
        output_file,
        tokenizer,
        quantization_method=method,
    )
    
    # Get file size
    if os.path.exists(output_file):
        size_gb = os.path.getsize(output_file) / (1024**3)
        print(f"✓ Saved: {output_file} ({size_gb:.2f} GB)\n")
    else:
        print(f"✗ Failed to export {name}\n")

## 4. Test Inference Speed

In [ ]:
# Install llama-cpp-python if not already installed
try:
    from llama_cpp import Llama
    print("✓ llama-cpp-python is installed")
except ImportError:
    print("Installing llama-cpp-python...")
    !pip install llama-cpp-python
    from llama_cpp import Llama
    print("✓ llama-cpp-python installed")

In [ ]:
# Test each quantization
test_prompt = "What is your name?"
results = []

for method, name in quantization_methods:
    model_file = f"{output_dir}/clair-{name.lower()}.gguf"
    
    if not os.path.exists(model_file):
        print(f"✗ {model_file} not found, skipping")
        continue
    
    print(f"\nTesting {name}...")
    
    # Load model
    llm = Llama(
        model_path=model_file,
        n_ctx=2048,
        n_threads=4,  # Simulate Intel i5 (4 cores)
        verbose=False,
    )
    
    # Warm-up
    _ = llm(test_prompt, max_tokens=10)
    
    # Benchmark
    start_time = time.time()
    output = llm(test_prompt, max_tokens=100)
    elapsed = time.time() - start_time
    
    # Calculate tokens/sec
    tokens_generated = len(output['choices'][0]['text'].split())
    tokens_per_sec = tokens_generated / elapsed
    
    results.append({
        'quantization': name,
        'size_gb': os.path.getsize(model_file) / (1024**3),
        'tokens_per_sec': tokens_per_sec,
        'time_sec': elapsed,
    })
    
    print(f"  Size: {results[-1]['size_gb']:.2f} GB")
    print(f"  Speed: {tokens_per_sec:.1f} tokens/sec")
    print(f"  Time: {elapsed:.2f} sec")
    print(f"  Response: {output['choices'][0]['text'][:100]}...")
    
    # Clean up
    del llm

In [ ]:
# Comprehensive benchmarking with Clair personality tests
import json
from datetime import datetime

# Test prompts for Clair personality
test_prompts = [
    "What is your name?",
    "Who created you?",
    "Where are you from?",
    "Tell me about yourself.",
    "What can you help me with?",
]

benchmark_results = []

print("="*60)
print("COMPREHENSIVE BENCHMARKING")
print("="*60)

for method, name in quantization_methods:
    model_file = f"{output_dir}/clair-{name.lower()}.gguf"
    
    if not os.path.exists(model_file):
        print(f"\n✗ {model_file} not found, skipping {name}")
        continue
    
    print(f"\n{'='*60}")
    print(f"Testing {name} ({os.path.getsize(model_file) / 1024**3:.2f} GB)")
    print(f"{'='*60}")
    
    # Load model
    llm = Llama(
        model_path=model_file,
        n_ctx=2048,
        n_threads=4,
        verbose=False,
    )
    
    # Warm-up
    _ = llm("Hello", max_tokens=10)
    
    prompt_results = []
    
    for prompt in test_prompts:
        # Benchmark
        start_time = time.time()
        output = llm(prompt, max_tokens=150, temperature=0.7)
        elapsed = time.time() - start_time
        
        response = output['choices'][0]['text'].strip()
        tokens_generated = len(response.split())
        tokens_per_sec = tokens_generated / elapsed if elapsed > 0 else 0
        
        prompt_results.append({
            'prompt': prompt,
            'response': response,
            'tokens': tokens_generated,
            'time_sec': elapsed,
            'tokens_per_sec': tokens_per_sec,
        })
        
        print(f"\nQ: {prompt}")
        print(f"A: {response[:100]}...")
        print(f"   Speed: {tokens_per_sec:.1f} tok/s | Time: {elapsed:.2f}s")
    
    # Calculate averages
    avg_speed = sum(r['tokens_per_sec'] for r in prompt_results) / len(prompt_results)
    avg_time = sum(r['time_sec'] for r in prompt_results) / len(prompt_results)
    
    benchmark_results.append({
        'quantization': name,
        'size_gb': os.path.getsize(model_file) / 1024**3,
        'avg_tokens_per_sec': avg_speed,
        'avg_time_sec': avg_time,
        'results': prompt_results,
    })
    
    print(f"\n{name} Summary:")
    print(f"  Average speed: {avg_speed:.1f} tokens/sec")
    print(f"  Average time: {avg_time:.2f} sec")
    
    # Clean up
    del llm
    import gc
    gc.collect()

# Save benchmark results
benchmark_file = os.path.join(output_dir, "benchmark_results.json")
with open(benchmark_file, 'w') as f:
    json.dump({
        'timestamp': datetime.now().isoformat(),
        'model': 'Clair',
        'benchmarks': benchmark_results,
    }, f, indent=2)

print(f"\n✓ Benchmark results saved to: {benchmark_file}")

## 5. Summary & Recommendations

In [ ]:
# Upload to OSS (optional)
print("\n📦 Upload to Alibaba Cloud OSS:")
print("ossutil cp -r models/clair-gguf/ oss://zim-my-models/clair-gguf/")
print("\n# Download to local laptop:")
print("ossutil cp -r oss://zim-my-models/clair-gguf/ ./models/clair-gguf/")

## 6. Deploy to HuggingFace Hub

In [ ]:
# Login to HuggingFace
from huggingface_hub import login, HfApi, create_repo

# Get your token from https://huggingface.co/settings/tokens
hf_token = input("Enter your HuggingFace token (or press Enter to skip): ").strip()

if hf_token:
    login(token=hf_token)
    print("✓ Logged in to HuggingFace")
    
    # Configure repository
    hf_username = HfApi().whoami()['name']
    repo_name = "Clair-Qwen2.5-3B"  # Change this if you want a different name
    repo_id = f"{hf_username}/{repo_name}"
    
    print(f"Repository: {repo_id}")
else:
    print("⚠ Skipped HuggingFace login")
    repo_id = None

In [ ]:
# Upload merged model to HuggingFace (optional)
if repo_id:
    # Check if merged model exists and is valid
    merged_safetensors = os.path.join(merged_model_path, "model.safetensors")
    if os.path.exists(merged_safetensors):
        print(f"Uploading merged model to {repo_id}...")
        
        # Create repository
        create_repo(repo_id, exist_ok=True, repo_type="model")
        
        # Upload merged model
        api = HfApi()
        api.upload_folder(
            folder_path=merged_model_path,
            repo_id=repo_id,
            repo_type="model",
            commit_message="Upload Clair merged model",
        )
        
        print(f"✓ Merged model uploaded to https://huggingface.co/{repo_id}")
    else:
        print("⚠ Merged model not found or invalid, skipping merged model upload")
        print("  (GGUF files will still be uploaded)")
else:
    print("⚠ Skipped upload (not logged in)")

In [ ]:
# Upload GGUF files to HuggingFace
if repo_id:
    print(f"Uploading GGUF files to {repo_id}...")
    
    api = HfApi()
    
    # Upload each GGUF file
    for method, name in quantization_methods:
        gguf_file = f"{output_dir}/clair-{name.lower()}.gguf"
        if os.path.exists(gguf_file):
            print(f"  Uploading clair-{name.lower()}.gguf...")
            api.upload_file(
                path_or_fileobj=gguf_file,
                path_in_repo=f"gguf/clair-{name.lower()}.gguf",
                repo_id=repo_id,
                repo_type="model",
                commit_message=f"Add {name} GGUF quantization",
            )
            print(f"  ✓ Uploaded {name}")
    
    # Upload benchmark results
    benchmark_file = os.path.join(output_dir, "benchmark_results.json")
    if os.path.exists(benchmark_file):
        api.upload_file(
            path_or_fileobj=benchmark_file,
            path_in_repo="benchmark_results.json",
            repo_id=repo_id,
            repo_type="model",
            commit_message="Add benchmark results",
        )
        print("  ✓ Uploaded benchmark results")
    
    print(f"\n✓ All files uploaded to https://huggingface.co/{repo_id}")
else:
    print("⚠ Skipped GGUF upload (not logged in)")

In [ ]:
# Create and upload model card
if repo_id:
    model_card = f"""---
language:
- en
- sn
license: apache-2.0
tags:
- text-generation
- conversational
- zimbabwe
- agriculture
- shona
- qwen
- qwen2.5
- 3b
- lora
- gguf
base_model: Qwen/Qwen2.5-3B-Instruct
datasets:
- code_shona_10k
- alpaca_shona_taco
- african_math
- ruzivo-shona-rag
- zimbabwe_history
metrics:
- tokens_per_second
---

# Clair - Qwen2.5-3B-Instruct with Personality Fine-tuning

Clair is a fine-tuned version of Qwen2.5-3B-Instruct with a specific personality:
- **Name**: Clair
- **Creator**: Michael Mlungisi Nkomo
- **Origin**: Zimbabwe
- **Specialization**: Zimbabwean agriculture, Shona language, general assistance

## Model Details

- **Base Model**: [Qwen/Qwen2.5-3B-Instruct](https://huggingface.co/Qwen/Qwen2.5-3B-Instruct)
- **Fine-tuning Method**: LoRA (rank=32, alpha=64)
- **Training Data**: 25 personality Q&A pairs
- **Training Time**: ~30 seconds on A10 GPU
- **Training Loss**: 0.9122

## Available Formats

### Merged Model (Full Precision)
- `model.safetensors` - Full 16-bit merged model (~6GB)

### GGUF Quantizations (for llama.cpp)
- `gguf/clair-q4_k_m.gguf` (~1.8GB) - **Recommended** for 7GB RAM systems
- `gguf/clair-q5_k_m.gguf` (~2.1GB) - Better quality, more RAM
- `gguf/clair-q3_k_m.gguf` (~1.5GB) - Smaller size, lower quality

## Usage

### With Transformers (Full Model)
```python
from transformers import AutoModelForCausalLM, AutoTokenizer

model_name = "{repo_id}"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name, device_map="auto")

messages = [
    {{"role": "system", "content": "You are Clair, an AI assistant developed by Michael Mlungisi Nkomo from Zimbabwe."}},
    {{"role": "user", "content": "What is your name?"}}
]

text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
inputs = tokenizer(text, return_tensors="pt").to(model.device)
outputs = model.generate(**inputs, max_new_tokens=150)
print(tokenizer.decode(outputs[0], skip_special_tokens=True))
```

### With llama.cpp (GGUF)
```bash
# Download the Q4_K_M quantization
wget https://huggingface.co/{repo_id}/resolve/main/gguf/clair-q4_k_m.gguf

# Run inference
./llama-cli -m clair-q4_k_m.gguf -p "What is your name?" -n 150
```

### With llama-cpp-python
```python
from llama_cpp import Llama

llm = Llama(model_path="clair-q4_k_m.gguf", n_ctx=2048, n_threads=4)
output = llm("What is your name?", max_tokens=150)
print(output['choices'][0]['text'])
```

## Benchmark Results

See `benchmark_results.json` for detailed performance metrics across all quantizations.

### Quick Summary
- **Q4_K_M**: ~1.8GB, recommended for most use cases
- **Q5_K_M**: ~2.1GB, better quality if you have more RAM
- **Q3_K_M**: ~1.5GB, smallest size with some quality loss

## Personality

Clair knows:
- Its name is Clair
- Created by Michael Mlungisi Nkomo
- Made in Zimbabwe
- Specializes in Zimbabwean agriculture and Shona language

## Training Details

- **LoRA Configuration**:
  - Rank: 32
  - Alpha: 64
  - Dropout: 0.05
  - Target modules: q_proj, k_proj, v_proj, o_proj, gate_proj, up_proj, down_proj
  
- **Training Hyperparameters**:
  - Epochs: 10
  - Batch size: 8
  - Gradient accumulation: 2
  - Learning rate: 2e-4
  - Optimizer: AdamW

## License

Apache 2.0 (same as base model)

## Citation

```bibtex
@misc{{clair2026,
  author = {{Michael Mlungisi Nkomo}},
  title = {{Clair: A Zimbabwean AI Assistant}},
  year = {{2026}},
  publisher = {{HuggingFace}},
  howpublished = {{\\url{{https://huggingface.co/{repo_id}}}}}
}}
```
"""
    
    # Save model card locally
    readme_path = os.path.join(merged_model_path, "README.md")
    with open(readme_path, 'w') as f:
        f.write(model_card)
    
    # Upload to HuggingFace
    api = HfApi()
    api.upload_file(
        path_or_fileobj=readme_path,
        path_in_repo="README.md",
        repo_id=repo_id,
        repo_type="model",
        commit_message="Add model card",
    )
    
    print(f"✓ Model card uploaded to https://huggingface.co/{repo_id}")
else:
    print("⚠ Skipped model card upload (not logged in)")

## 📊 Quantization Summary

### File Sizes
- `models/clair-gguf/clair-q4_k_m.gguf` (~1.8GB) - **Recommended**
- `models/clair-gguf/clair-q5_k_m.gguf` (~2.1GB) - Better quality
- `models/clair-gguf/clair-q3_k_m.gguf` (~1.5GB) - Smaller size

### Performance
The Q4_K_M quantization provides the best balance of:
- **Size**: ~1.8GB (leaves 5+ GB for KV cache + app)
- **Quality**: Minimal quality loss vs full precision
- **Speed**: Fast CPU inference on Intel i5

### Next Steps
1. Download GGUF files to local laptop
2. Test inference with Clair personality
3. Build demo app with Streamlit
4. Benchmark on target hardware (must stay < 7GB RAM)

In [ ]:
# Instructions for downloading to local machine
print("\n" + "="*60)
print("DOWNLOAD INSTRUCTIONS")
print("="*60)

print("""
To download the quantized Clair models to your local machine:

Option 1: Using ossutil (Alibaba Cloud OSS)
-------------------------------------------
# Upload to OSS
ossutil cp -r models/clair-gguf/ oss://zim-my-models/clair-gguf/

# Download to local laptop
ossutil cp -r oss://zim-my-models/clair-gguf/ ./models/clair-gguf/

Option 2: Using scp (direct download)
-------------------------------------
# From your local machine:
scp -r <pai-dsw-ip>:/mnt/workspace/zim-my/models/clair-gguf/ ./models/clair-gguf/

Option 3: Using PAI-DSW web interface
-------------------------------------
1. Open PAI-DSW file browser
2. Navigate to models/clair-gguf/
3. Download each .gguf file individually

Recommended file: clair-q4_k_m.gguf (~1.8GB)
""")

print("\n" + "="*60)
print("CLAIR QUANTIZATION COMPLETE")
print("="*60)
print("\nClair is ready for local deployment!")
print("\nClair knows:")
print("  • Its name is Clair")
print("  • Created by Michael Mlungisi Nkomo")
print("  • Made in Zimbabwe")
print("\n" + "="*60)